In [ ]:
# ============================================================================
# STEP 1: INSTALLATION
# ============================================================================
print("Installing required packages...")
!pip install -q transformers datasets peft accelerate bitsandbytes trl torch sentencepiece

print("\nInstallation complete!")

In [ ]:
# ============================================================================
# STEP 2: IMPORTS
# ============================================================================
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import load_dataset
import json
import os

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# ============================================================================
# STEP 3: LOAD DATASET (FULL 18K SAMPLES)
# ============================================================================
print("\nLoading Python Code Instructions dataset (FULL VERSION)...")

# Load the complete dataset (18k examples)
dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")

print(f"Dataset loaded: {len(dataset)} samples (FULL DATASET)")
print("\nExample:")
print(dataset[0])

In [ ]:
# ============================================================================
# STEP 4: PREPARE DATA FOR TRAINING
# ============================================================================
print("\nPreparing data...")

def format_instruction(example):
    """Format the instruction and output into a single text"""
    instruction = example["instruction"]
    output = example["output"]

    # Alpaca-style formatting
    text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}\n\n"

    return {"text": text}

# Format all examples
dataset = dataset.map(format_instruction, remove_columns=dataset.column_names)

# Split into train/validation (90/10 split)
train_test = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test["train"]
eval_dataset = train_test["test"]

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")

In [ ]:
# ============================================================================
# STEP 5: LOAD BASE MODEL
# ============================================================================
print("\nLoading CodeGen-350M model...")

model_name = "Salesforce/codegen-350M-mono"  # 350M params, Python-focused
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # Use FP16 for efficiency
)

# Set padding token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded!")
print(f"Model parameters: {model.num_parameters() / 1e6:.0f}M")

# Move to GPU
if torch.cuda.is_available():
    model = model.to("cuda")

In [ ]:
# ============================================================================
# STEP 6: TOKENIZE DATASET
# ============================================================================
print("\nTokenizing dataset (this may take a few minutes)...")

def tokenize_function(examples):
    """Tokenize the text"""
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

# Tokenize in batches
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,  # Parallel processing
    remove_columns=train_dataset.column_names,
    desc="Tokenizing training data"
)

tokenized_eval = eval_dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=eval_dataset.column_names,
    desc="Tokenizing eval data"
)

print("Tokenization complete!")

In [ ]:
# ============================================================================
# STEP 7: CONFIGURE LORA (FIXED FOR CODEGEN)
# ============================================================================
print("\nConfiguring LoRA with PRODUCTION settings...")

# First, let's check what modules are available in this model
print("\nInspecting model architecture...")
for name, module in model.named_modules():
    if 'attn' in name.lower() or 'proj' in name.lower():
        print(f"  {name}: {type(module).__name__}")

# For CodeGen models, we target: "qkv_proj" instead of "c_attn"
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,             # 2x rank (standard practice)
    target_modules=["qkv_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("LoRA configured!")
print("Target modules: qkv_proj (attention projection layers)")

In [ ]:
# ============================================================================
# STEP 8: TRAINING CONFIGURATION (FIXED)
# ============================================================================
print("\nSetting up PRODUCTION training configuration...")

training_args = TrainingArguments(
    output_dir="./lora_production",
    num_train_epochs=6,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,         # Effective batch = 16
    learning_rate=2e-4,
    fp16=True,                             # Use FP16 for speed
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    warmup_steps=500,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    report_to="none",
    logging_first_step=True,
    optim="adamw_torch"
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator
)

print("Trainer ready!")
print(f"\nEstimated training time: 1-2 hours")
print(f"Total training steps: {len(tokenized_train) // (4 * 4) * 6}")

In [ ]:
# ============================================================================
# STEP 9: TRAIN THE MODEL
# ============================================================================

print("STARTING PRODUCTION TRAINING")

print("This will take 1-2 hours depending on your GPU.")
print("Training on full 18k samples with 6 epochs.")

# Train!
trainer.train()

print("TRAINING COMPLETE!")


In [ ]:
# ============================================================================
# STEP 10: EVALUATE THE MODEL
# ============================================================================
print("\nEvaluating final model...")

eval_results = trainer.evaluate()
print(f"\nFinal Evaluation Loss: {eval_results['eval_loss']:.4f}")

In [ ]:
# ============================================================================
# STEP 11: SAVE LORA ADAPTERS
# ============================================================================
print("\nSaving LoRA adapters...")

# Save adapters
output_dir = "./lora_adapters"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"LoRA adapters saved to {output_dir}/")

# Save training info
training_info = {
    "model": model_name,
    "dataset": "iamtarun/python_code_instructions_18k_alpaca",
    "num_samples": len(train_dataset),
    "epochs": 6,
    "lora_r": 32,
    "lora_alpha": 64,
    "final_eval_loss": float(eval_results['eval_loss']),
    "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "total_params": sum(p.numel() for p in model.parameters())
}

with open(f"{output_dir}/training_info.json", "w") as f:
    json.dump(training_info, f, indent=2)

print("\nTraining info saved!")

In [ ]:
# ============================================================================
# STEP 12: TEST THE MODEL
# ============================================================================
print("\nTesting the fine-tuned model with various examples...")

def generate_code(instruction, max_length=300):
    """Generate code for a given instruction"""
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt")

    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=1,
        temperature=0.2,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.1
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract just the response part
    if "### Response:" in response:
        code = response.split("### Response:")[1].strip()
        return code
    return response

# Test with diverse examples
test_instructions = [
    "Write a Python function to calculate the factorial of a number using recursion",
    "Create a function to check if a string is a palindrome",
    "Write code to merge two sorted lists into one sorted list",
    "Implement a binary search algorithm in Python",
    "Create a class to represent a stack data structure with push and pop operations",
]


print("TEST RESULTS")


for i, instruction in enumerate(test_instructions, 1):

    print(f"Test {i}/{len(test_instructions)}")
    print(f"Instruction: {instruction}")
    print(f"\nGenerated Code:")

    code = generate_code(instruction, max_length=400)
    print(code)

In [ ]:
# ============================================================================
# STEP 13: DOWNLOAD INSTRUCTIONS
# ============================================================================
print("DOWNLOAD INSTRUCTIONS")
print("""
1. In Colab, look at the left sidebar (folder icon)
2. Navigate to 'lora_adapters' folder
3. Right-click on the folder -> Download
4. Extract the ZIP file on your computer

Files included:
  - adapter_config.json (LoRA configuration)
  - adapter_model.safetensors (LoRA weights)
  - tokenizer files
  - training_info.json (training metadata)

You now have a PRODUCTION-QUALITY fine-tuned model!
""")

In [ ]:
# ============================================================================
# BONUS: CREATE ZIP FILE
# ============================================================================
print("\nCreating ZIP file for easy download...")
!zip -r lora_adapters.zip lora_adapters/

print("ALL DONE!")
print("""
Download 'lora_adapters.zip' from the files panel!

This model was trained on:
- 18,000 Python code examples
- 350M parameter base model (CodeGen-350M-mono)
- 6 epochs of training
- Production-grade LoRA configuration (r=32, alpha=64)

It should generate HIGH-QUALITY Python code!
""")